# Bank XYZ — Preprocessing & Feature Engineering
Dashboard Customer Satisfaction Survey

**Output:** `data/processed_bankxyz.csv` siap dipakai dashboard Streamlit

---
### Struktur Notebook
1. Load & inspect data
2. Parsing skala teks → numerik
3. Feature engineering: NPS, IPA, Emotion Index, Segmentasi, Brand Perception
4. Agregasi per cabang, provinsi, demografi
5. Export ke CSV

In [48]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Path absolut (sesuaikan jika folder berbeda) ─────────────
DATA_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Load raw data ─────────────────────────────────────────────
FILE = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data\Deka_project_dataset_BankXYZ.xlsx'
df = pd.read_excel(FILE, header=1)
print(f'Shape: {df.shape}')
print(f'Rows: {len(df):,} | Columns: {df.shape[1]}')


Shape: (1730, 633)
Rows: 1,730 | Columns: 633


## 1. Mapping Kolom Penting

In [49]:
# ── Column mapping ───────────────────────────────────────────
# Identitas & lokasi
COL_ID       = 'SERIAL'
COL_PROV     = 'PROV'
COL_KOTA     = 'KABKOTA'
COL_CABANG   = 'CABANG'
COL_PANEL    = 'PANEL'   # Teller / CS

# Demografi
COL_GENDER   = 'S1'
COL_AGE_RAW  = 'S2_1'
COL_AGE_GRP  = 'S2_2'
COL_LAMA_NAS = 'S4'
COL_FREK     = 'S7'
COL_NIKAH    = 'P1'
COL_ANAK     = 'P2'
COL_PENDIDIK = 'P3'
COL_PEKERJAAN= 'P4'
COL_PENGELUAR= 'P5'
COL_PENGHASIL= 'P6'

# KPI utama XYZ
COL_NPS_XYZ      = 'G1A'
COL_CSI_XYZ      = 'E1A'
COL_LOYALTY_XYZ  = 'F1A'

# NPS alasan (open-ended)
COL_NPS_REASON   = 'G1B'

# Overall satisfaction per kategori layanan - XYZ
COL_OVR_OPERASIONAL = 'T_KC2_107'
COL_OVR_PARKIR      = 'T_KC2_110'
COL_OVR_BANKING_HALL= 'T_KC2_113'
COL_OVR_TOILET      = 'T_KC2_116'
COL_OVR_SEKURITI    = 'T_SC2_47'
COL_OVR_TELLER      = 'T_TL3_59'
COL_OVR_CS          = 'T_CS3_71'
COL_OVR_ATM         = 'T_AT3_56'

# IPA: Kantor Cabang - Importance (mod3==2) / Satisfaction (mod3==0)
IPA_KC_IMP = [c for c in df.columns if str(c).startswith('T_KC2_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
IPA_KC_SAT = [c for c in df.columns if str(c).startswith('T_KC2_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
              and int(str(c).split('_')[-1]) <= 105]

# IPA: Sekuriti
IPA_SC_IMP = [c for c in df.columns if str(c).startswith('T_SC2_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
IPA_SC_SAT = [c for c in df.columns if str(c).startswith('T_SC2_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
              and int(str(c).split('_')[-1]) <= 45]

# IPA: Teller
IPA_TL_IMP = [c for c in df.columns if str(c).startswith('T_TL3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
IPA_TL_SAT = [c for c in df.columns if str(c).startswith('T_TL3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
              and int(str(c).split('_')[-1]) <= 57]

# IPA: Customer Service
IPA_CS_IMP = [c for c in df.columns if str(c).startswith('T_CS3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
IPA_CS_SAT = [c for c in df.columns if str(c).startswith('T_CS3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
              and int(str(c).split('_')[-1]) <= 69]

# IPA: ATM
IPA_AT_IMP = [c for c in df.columns if str(c).startswith('T_AT3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
IPA_AT_SAT = [c for c in df.columns if str(c).startswith('T_AT3_') and
              str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
              and int(str(c).split('_')[-1]) <= 55]

# Emotion - XYZ (positif & negatif)
EMOTION_XYZ_POSITIVE = ['T_I1A_2','T_I1A_5','T_I1A_8','T_I1A_11',
                         'T_I1A_14','T_I1A_17','T_I1A_20','T_I1A_23','T_I1A_26']
EMOTION_XYZ_NEGATIVE = ['T_I1A_29','T_I1A_32','T_I1A_35','T_I1A_38',
                         'T_I1A_41','T_I1A_44','T_I1A_47']
EMOTION_LABELS = {
    'T_I1A_2':'Bahagia','T_I1A_5':'Percaya','T_I1A_8':'Dihargai',
    'T_I1A_11':'Diperhatikan','T_I1A_14':'Aman','T_I1A_17':'Fokus',
    'T_I1A_20':'Dimanjakan','T_I1A_23':'Tertarik','T_I1A_26':'Semangat',
    'T_I1A_29':'Tidak Puas','T_I1A_32':'Frustasi','T_I1A_35':'Kecewa',
    'T_I1A_38':'Tertekan','T_I1A_41':'Tidak Bahagia',
    'T_I1A_44':'Diabaikan','T_I1A_47':'Tergesa-gesa'
}

# Brand perception XYZ vs Kompetitor
BRAND_XYZ_COLS = [c for c in df.columns if str(c).startswith('T_H1A_') and
                  str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2]
BRAND_KOMP_COLS= [c for c in df.columns if str(c).startswith('T_H1A_') and
                  str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0
                  and int(str(c).split('_')[-1]) <= 45]

# Digitalisasi cabang
DIGI_COLS = [c for c in df.columns if str(c).startswith('T_SL1_')]

print('Mapping kolom selesai')
print(f'IPA KC: {len(IPA_KC_IMP)} importance / {len(IPA_KC_SAT)} satisfaction')
print(f'IPA SC: {len(IPA_SC_IMP)} / {len(IPA_SC_SAT)}')
print(f'IPA TL: {len(IPA_TL_IMP)} / {len(IPA_TL_SAT)}')
print(f'IPA CS: {len(IPA_CS_IMP)} / {len(IPA_CS_SAT)}')
print(f'IPA AT: {len(IPA_AT_IMP)} / {len(IPA_AT_SAT)}')
print(f'Emotion XYZ: {len(EMOTION_XYZ_POSITIVE)} positif / {len(EMOTION_XYZ_NEGATIVE)} negatif')

Mapping kolom selesai
IPA KC: 39 importance / 35 satisfaction
IPA SC: 16 / 15
IPA TL: 20 / 19
IPA CS: 24 / 23
IPA AT: 19 / 18
Emotion XYZ: 9 positif / 7 negatif


## 2. Parsing Skala Teks → Numerik

In [50]:
def parse_scale(val):
    """Konversi '6  SANGAT PUAS', '5', 999 → float. Return NaN jika invalid."""
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ['', ' ', '999']: return np.nan
    try:
        first = s[0]
        if first.isdigit():
            return float(first)
    except:
        pass
    return np.nan

def parse_nps(val):
    """Khusus NPS: '10  PASTI AKAN...' → 10, '9' → 9"""
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.startswith('10'): return 10.0
    try:
        first = s[0]
        if first.isdigit(): return float(first)
    except:
        pass
    return np.nan

# Parse KPI utama
df['nps_num']     = df[COL_NPS_XYZ].apply(parse_nps)
df['csi_num']     = df[COL_CSI_XYZ].apply(parse_scale)
df['loyalty_num'] = df[COL_LOYALTY_XYZ].apply(parse_scale)

# NPS segmentation
def nps_segment(v):
    if pd.isna(v): return np.nan
    if v >= 9: return 'Promoter'
    if v >= 7: return 'Passive'
    return 'Detractor'

df['nps_segment'] = df['nps_num'].apply(nps_segment)

# NPS score function
def nps_score(series):
    s = series.dropna()
    if len(s) == 0: return np.nan
    return round(((s >= 9).sum() - (s <= 6).sum()) / len(s) * 100, 1)

# Parse overall satisfaction per kategori
OVERALL_COLS = {
    'ovr_operasional': COL_OVR_OPERASIONAL,
    'ovr_parkir':      COL_OVR_PARKIR,
    'ovr_banking_hall':COL_OVR_BANKING_HALL,
    'ovr_toilet':      COL_OVR_TOILET,
    'ovr_sekuriti':    COL_OVR_SEKURITI,
    'ovr_teller':      COL_OVR_TELLER,
    'ovr_cs':          COL_OVR_CS,
}
for newcol, rawcol in OVERALL_COLS.items():
    if rawcol in df.columns:
        df[newcol] = df[rawcol].apply(parse_scale)

# Parse IPA cols
ALL_IPA_COLS = (IPA_KC_IMP + IPA_KC_SAT + IPA_SC_IMP + IPA_SC_SAT +
                IPA_TL_IMP + IPA_TL_SAT + IPA_CS_IMP + IPA_CS_SAT +
                IPA_AT_IMP + IPA_AT_SAT)
for c in ALL_IPA_COLS:
    if c in df.columns:
        df[c+'_num'] = df[c].apply(parse_scale)

# Parse emotion cols
ALL_EMOTION_COLS = EMOTION_XYZ_POSITIVE + EMOTION_XYZ_NEGATIVE
for c in ALL_EMOTION_COLS:
    if c in df.columns:
        df[c+'_num'] = df[c].apply(parse_scale)

print('Parsing selesai')
print(f'NPS valid: {df["nps_num"].notna().sum()} / {len(df)}')
print(f'CSI valid: {df["csi_num"].notna().sum()} / {len(df)}')
print(f'Loyalty valid: {df["loyalty_num"].notna().sum()} / {len(df)}')
print(f'\nNPS Distribution:')
print(df['nps_segment'].value_counts())
print(f'\nNPS Score: {nps_score(df["nps_num"])}')

Parsing selesai
NPS valid: 1730 / 1730
CSI valid: 1730 / 1730
Loyalty valid: 1730 / 1730

NPS Distribution:
nps_segment
Promoter     1423
Passive       283
Detractor      24
Name: count, dtype: int64

NPS Score: 80.9


## 3. Feature Engineering

In [51]:
# ── 3A. Emotion Index ─────────────────────────────────────────
pos_cols_num = [c+'_num' for c in EMOTION_XYZ_POSITIVE if c in df.columns]
neg_cols_num = [c+'_num' for c in EMOTION_XYZ_NEGATIVE if c in df.columns]

df['emotion_positive_score'] = df[pos_cols_num].mean(axis=1)
df['emotion_negative_score'] = df[neg_cols_num].mean(axis=1)

# Emotion Net Score: positif - negatif (normalized ke -5 to 5)
df['emotion_net'] = df['emotion_positive_score'] - df['emotion_negative_score']

print('Emotion Index:')
print(f'  Positive avg: {df["emotion_positive_score"].mean():.2f}')
print(f'  Negative avg: {df["emotion_negative_score"].mean():.2f}')
print(f'  Net avg:      {df["emotion_net"].mean():.2f}')

Emotion Index:
  Positive avg: 5.78
  Negative avg: 1.58
  Net avg:      4.20


In [52]:
# ── 3B. IPA Matrix per Kategori ───────────────────────────────
# Importance Performance Analysis: rata-rata importance vs satisfaction per atribut
def build_ipa(imp_cols, sat_cols, prefix, df):
    """Build IPA dataframe untuk satu kategori touchpoint."""
    records = []
    n = min(len(imp_cols), len(sat_cols))
    for i in range(n):
        ic = imp_cols[i]
        sc = sat_cols[i]
        ic_num = ic + '_num'
        sc_num = sc + '_num'
        if ic_num in df.columns and sc_num in df.columns:
            imp_mean = df[ic_num].mean()
            sat_mean = df[sc_num].mean()
            if not (np.isnan(imp_mean) or np.isnan(sat_mean)):
                records.append({
                    'kategori': prefix,
                    'atribut_idx': i+1,
                    'col_imp': ic,
                    'col_sat': sc,
                    'importance': round(imp_mean, 3),
                    'performance': round(sat_mean, 3),
                    'gap': round(sat_mean - imp_mean, 3),
                })
    return pd.DataFrame(records)
ipa_kc = build_ipa(IPA_KC_IMP, IPA_KC_SAT, 'Kantor Cabang', df)
ipa_sc = build_ipa(IPA_SC_IMP, IPA_SC_SAT, 'Sekuriti', df)
ipa_tl = build_ipa(IPA_TL_IMP, IPA_TL_SAT, 'Teller', df)
ipa_cs = build_ipa(IPA_CS_IMP, IPA_CS_SAT, 'Customer Service', df)
ipa_at = build_ipa(IPA_AT_IMP, IPA_AT_SAT, 'ATM', df)
ipa_all = pd.concat([ipa_kc, ipa_sc, ipa_tl, ipa_cs, ipa_at], ignore_index=True)
# Kuadran IPA
imp_median = ipa_all['importance'].median()
sat_median = ipa_all['performance'].median()
def ipa_quadrant(row):
    hi = row['importance'] >= imp_median
    good = row['performance'] >= sat_median
    if hi and good:     return 'Keep Up (pertahankan)'
    if hi and not good: return 'Quick Win (prioritas perbaikan)'
    if not hi and good: return 'Possible Overkill'
    return 'Low Priority'
ipa_all['kuadran'] = ipa_all.apply(ipa_quadrant, axis=1)
ipa_all.to_csv(f'{DATA_DIR}/ipa_matrix.csv', index=False)
print(f'IPA Matrix: {len(ipa_all)} atribut')
print(ipa_all['kuadran'].value_counts())
print(f'\nQuick Wins (prioritas):')
print(ipa_all[ipa_all['kuadran']=='Quick Win (prioritas perbaikan)'][['kategori','atribut_idx','importance','performance','gap']].head(10))

IPA Matrix: 110 atribut
kuadran
Keep Up (pertahankan)              42
Low Priority                       41
Quick Win (prioritas perbaikan)    14
Possible Overkill                  13
Name: count, dtype: int64

Quick Wins (prioritas):
         kategori  atribut_idx  importance  performance    gap
10  Kantor Cabang           11       5.882        5.401 -0.481
11  Kantor Cabang           12       5.867        5.352 -0.515
16  Kantor Cabang           17       5.869        5.381 -0.488
26  Kantor Cabang           27       5.888        5.350 -0.538
27  Kantor Cabang           28       5.868        5.330 -0.539
32  Kantor Cabang           33       5.873        5.374 -0.500
48       Sekuriti           14       5.940        5.399 -0.541
52         Teller            3       5.880        5.395 -0.485
64         Teller           15       5.872        5.404 -0.468
66         Teller           17       5.869        5.378 -0.491


In [53]:
# ── 3C. Customer Segmentation ─────────────────────────────────
# Segmentasi berdasarkan: lama nasabah, frekuensi transaksi, NPS

LAMA_MAP = {
    'Kurang dari 1 tahun': 0,
    '1 - 2 tahun': 1,
    '2 - 3 tahun': 2,
    '3 - 5 tahun': 3,
    '5 tahun atau lebih': 4,
}
FREK_MAP = {
    'Kurang dari 1 kali sebulan': 0,
    '1 - 2 kali sebulan': 1,
    '1 minggu 1 kali': 2,
    '1 minggu 2 kali atau lebih': 3,
}

df['lama_num'] = df[COL_LAMA_NAS].map(LAMA_MAP)
df['frek_num'] = df[COL_FREK].map(FREK_MAP)

# Segmen: High Value, Regular, At Risk, New
def customer_segment(row):
    nps = row['nps_num']
    lama = row['lama_num']
    frek = row['frek_num']
    if pd.isna(nps) or pd.isna(lama): return 'Unknown'
    if nps >= 9 and lama >= 3 and frek >= 2:  return 'Loyal Champion'
    if nps >= 9 and lama >= 2:                 return 'Satisfied'
    if nps <= 6 and lama >= 3:                 return 'At Risk'
    if lama <= 1:                               return 'New Customer'
    return 'Passive'

df['customer_segment'] = df.apply(customer_segment, axis=1)
print('Customer Segmentation:')
print(df['customer_segment'].value_counts())

Customer Segmentation:
customer_segment
Unknown           847
Satisfied         434
Loyal Champion    316
Passive           124
At Risk             9
Name: count, dtype: int64


In [54]:
# ── 3D. Brand Perception ──────────────────────────────────────
# % setuju (scale >= 4) per atribut brand XYZ vs kompetitor
BRAND_LABELS = [
    'Bank terkenal','Digunakan banyak orang','Membuat merasa aman',
    'Membuat merasa dihargai','Banyak ATM','Banyak cabang',
    'Reputasi baik','Produk/layanan lengkap','Layanan online informasi',
    'Banyak channel keluhan','Menguntungkan investasi','Membuat bangga',
    'Percaya diri bertransaksi','Bergengsi/prestise','Mudah transaksi kapanpun',
]
brand_records = []
for i, label in enumerate(BRAND_LABELS):
    xyz_col  = BRAND_XYZ_COLS[i]  if i < len(BRAND_XYZ_COLS)  else None
    komp_col = BRAND_KOMP_COLS[i] if i < len(BRAND_KOMP_COLS) else None
    xyz_agree = komp_agree = np.nan
    if xyz_col and xyz_col in df.columns:
        vals = df[xyz_col].apply(parse_scale).dropna()
        if len(vals) > 0:
            xyz_agree = round((vals >= 4).mean() * 100, 1)
    if komp_col:
        # parse kompetitor
        komp_num = df[komp_col].apply(parse_scale) if komp_col in df.columns else pd.Series(dtype=float)
        vals_k = komp_num.dropna()
        if len(vals_k) > 0:
            komp_agree = round((vals_k >= 4).mean() * 100, 1)
    brand_records.append({
        'atribut': label,
        'xyz_pct_agree': xyz_agree,
        'komp_pct_agree': komp_agree,
        'selisih': round(xyz_agree - komp_agree, 1) if not (np.isnan(xyz_agree) or np.isnan(komp_agree)) else np.nan
    })
brand_df = pd.DataFrame(brand_records)
brand_df.to_csv(f'{DATA_DIR}/brand_perception.csv', index=False)
print('Brand Perception (% setuju):')
print(brand_df[['atribut','xyz_pct_agree','komp_pct_agree','selisih']].head(10))

Brand Perception (% setuju):
                    atribut  xyz_pct_agree  komp_pct_agree  selisih
0             Bank terkenal           99.9            97.1      2.8
1    Digunakan banyak orang           99.8            97.8      2.0
2       Membuat merasa aman           99.9            97.8      2.1
3   Membuat merasa dihargai           99.8            96.3      3.5
4                Banyak ATM           99.9            96.2      3.7
5             Banyak cabang           99.7            96.2      3.5
6             Reputasi baik           99.8            97.4      2.4
7    Produk/layanan lengkap          100.0            98.0      2.0
8  Layanan online informasi          100.0            98.0      2.0
9    Banyak channel keluhan           99.9            97.3      2.6


In [55]:
# ── 3E. Switching Analysis ────────────────────────────────────
# Bank utama simpan vs bank utama transaksi

COL_BANK_SIMPAN    = 'A1B'   # Bank utama simpan dana
COL_BANK_TRANSAKSI = 'A1C'   # Bank utama transaksi

if COL_BANK_SIMPAN in df.columns:
    simpan_counts = df[COL_BANK_SIMPAN].value_counts(normalize=True).mul(100).round(1)
    print('Bank Utama Simpan (top 5):')
    print(simpan_counts.head())

if COL_BANK_TRANSAKSI in df.columns:
    transaksi_counts = df[COL_BANK_TRANSAKSI].value_counts(normalize=True).mul(100).round(1)
    print('\nBank Utama Transaksi (top 5):')
    print(transaksi_counts.head())

Bank Utama Simpan (top 5):
A1B
Bank XYZ                       91.2
Bank Central Asia (BCA)         2.8
Bank Rakyat Indonesia (BRI)     1.8
Bank Mandiri                    1.8
Bank Negara Indonesia (BNI)     1.0
Name: proportion, dtype: float64

Bank Utama Transaksi (top 5):
A1C
Bank XYZ                       88.6
Bank Central Asia (BCA)         4.9
Bank Mandiri                    2.3
Bank Rakyat Indonesia (BRI)     2.3
Bank Negara Indonesia (BNI)     1.0
Name: proportion, dtype: float64


## 4. Agregasi untuk Dashboard

In [56]:
# ── 4A. Agregasi per Cabang ───────────────────────────────────
overall_num_cols = list(OVERALL_COLS.keys())
agg_dict = {
    'nps_num': nps_score,
    'csi_num': 'mean',
    'loyalty_num': 'mean',
    'emotion_positive_score': 'mean',
    'emotion_negative_score': 'mean',
    'emotion_net': 'mean',
    'SERIAL': 'count',
}
for c in overall_num_cols:
    if c in df.columns:
        agg_dict[c] = 'mean'
branch_agg = df.groupby([COL_PROV, COL_KOTA, COL_CABANG]).agg(agg_dict).reset_index()
branch_agg.rename(columns={'SERIAL':'n_responden','nps_num':'nps_score'}, inplace=True)
branch_agg['nps_score'] = branch_agg['nps_score'].round(1)
branch_agg['csi_num']   = branch_agg['csi_num'].round(3)
branch_agg['loyalty_num'] = branch_agg['loyalty_num'].round(3)
branch_agg.to_csv(f'{DATA_DIR}/agg_branch.csv', index=False)
print(f'Agregasi cabang: {len(branch_agg)} cabang')
print(branch_agg[[COL_PROV, COL_CABANG, 'nps_score', 'csi_num', 'n_responden']].head(10))

Agregasi cabang: 128 cabang
     PROV        CABANG  nps_score  csi_num  n_responden
0    Bali    Denpasar 1       94.4    6.000           18
1    Bali    Denpasar 2      100.0    6.000            9
2  Banten     Cilegon 1       77.8    5.889            9
3  Banten     Cilegon 2       94.4    6.000           18
4  Banten       Lebak 1      100.0    6.000           18
5  Banten       Lebak 2       44.4    5.444            9
6  Banten  Pandeglang 1       94.4    6.000           18
7  Banten  Pandeglang 2       88.9    6.000            9
8  Banten  Pandeglang 3      100.0    6.000           18
9  Banten  Pandeglang 4       88.9    5.889            9


In [57]:
# ── 4B. Agregasi per Provinsi ─────────────────────────────────
prov_agg = df.groupby(COL_PROV).agg(agg_dict).reset_index()
prov_agg.rename(columns={'SERIAL':'n_responden','nps_num':'nps_score'}, inplace=True)
prov_agg['nps_score'] = prov_agg['nps_score'].round(1)
prov_agg['csi_num']   = prov_agg['csi_num'].round(3)
prov_agg.to_csv(f'{DATA_DIR}/agg_provinsi.csv', index=False)
print('Agregasi per provinsi:')
print(prov_agg[[COL_PROV,'nps_score','csi_num','n_responden']].sort_values('nps_score', ascending=False))

Agregasi per provinsi:
                  PROV  nps_score  csi_num  n_responden
11    Sulawesi Selatan      100.0    6.000           27
0                 Bali       96.3    6.000           27
6   Kalimantan Selatan       92.6    6.000           27
3           Jawa Barat       85.9    5.926          945
12    Sumatera Selatan       85.2    5.963           27
1               Banten       82.9    5.921          216
10                Riau       77.8    5.944           18
9              Lampung       77.8    5.778           27
4          Jawa Tengah       72.7    5.758           99
5           Jawa Timur       66.7    5.889           27
2          DKI Jakarta       65.1    5.784          218
7     Kalimantan Timur       59.3    5.556           27
8       Kepulauan Riau       55.6    5.704           27
13      Sumatera Utara       50.0    5.889           18


In [58]:
# ── 4C. Agregasi per Demografi ────────────────────────────────
# Gender
gender_agg = df.groupby(COL_GENDER).agg({'nps_num': nps_score, 'csi_num':'mean', 'loyalty_num':'mean', 'SERIAL':'count'}).reset_index()
gender_agg.columns = ['gender','nps_score','csi_mean','loyalty_mean','n']
gender_agg.to_csv(f'{DATA_DIR}/agg_gender.csv', index=False)
# Usia
usia_agg = df.groupby(COL_AGE_GRP).agg({'nps_num': nps_score, 'csi_num':'mean', 'loyalty_num':'mean', 'SERIAL':'count'}).reset_index()
usia_agg.columns = ['usia_group','nps_score','csi_mean','loyalty_mean','n']
usia_agg.to_csv(f'{DATA_DIR}/agg_usia.csv', index=False)
# Panel
panel_agg = df.groupby(COL_PANEL).agg({'nps_num': nps_score, 'csi_num':'mean', 'loyalty_num':'mean', 'SERIAL':'count'}).reset_index()
panel_agg.columns = ['panel','nps_score','csi_mean','loyalty_mean','n']
panel_agg.to_csv(f'{DATA_DIR}/agg_panel.csv', index=False)
# Segmen
seg_agg = df.groupby('customer_segment').agg({'nps_num': nps_score, 'csi_num':'mean', 'SERIAL':'count'}).reset_index()
seg_agg.columns = ['segmen','nps_score','csi_mean','n']
seg_agg.to_csv(f'{DATA_DIR}/agg_segmen.csv', index=False)
print('Gender:')
print(gender_agg)
print('\nPanel:')
print(panel_agg)
print('\nSegmen:')
print(seg_agg)

Gender:
   gender  nps_score  csi_mean  loyalty_mean    n
0    Pria       81.5  5.907027      5.885405  925
1  Wanita       80.1  5.869565      5.860870  805

Panel:
                panel  nps_score  csi_mean  loyalty_mean    n
0      CS (KUOTA 50%)       79.8  5.882081      5.873988  865
1  Teller (KUOTA 50%)       82.0  5.897110      5.873988  865

Segmen:
           segmen  nps_score  csi_mean    n
0         At Risk     -100.0  5.555556    9
1  Loyal Champion      100.0  5.965190  316
2         Passive        0.0  5.677419  124
3       Satisfied      100.0  5.965438  434
4         Unknown       77.7  5.857143  847


In [59]:
# ── 4D. Emotion Summary ───────────────────────────────────────
emo_records = []
for col, label in EMOTION_LABELS.items():
    num_col = col + '_num'
    if num_col in df.columns:
        vals = df[num_col].dropna()
        if len(vals) > 0:
            tipe = 'positif' if col in EMOTION_XYZ_POSITIVE else 'negatif'
            pct_high = round((vals >= 5).mean() * 100, 1)  # % responden merasakan kuat (>=5)
            emo_records.append({
                'emosi': label,
                'tipe': tipe,
                'mean_score': round(vals.mean(), 3),
                'pct_strong': pct_high,
                'n': len(vals),
            })
emo_df = pd.DataFrame(emo_records)
emo_df.to_csv(f'{DATA_DIR}/emotion_summary.csv', index=False)
print('Emotion Summary:')
print(emo_df.sort_values('mean_score', ascending=False))

Emotion Summary:
            emosi     tipe  mean_score  pct_strong     n
4            Aman  positif       5.826        99.0  1730
1         Percaya  positif       5.820        98.8  1730
2        Dihargai  positif       5.810        98.6  1730
3    Diperhatikan  positif       5.804        98.6  1730
8        Semangat  positif       5.780        97.7  1730
0         Bahagia  positif       5.779        97.9  1730
6      Dimanjakan  positif       5.747        96.9  1730
5           Fokus  positif       5.740        97.5  1730
7        Tertarik  positif       5.724        97.0  1730
15   Tergesa-gesa  negatif       1.666        10.4  1730
12       Tertekan  negatif       1.629        10.9  1730
14      Diabaikan  negatif       1.614        10.7  1730
13  Tidak Bahagia  negatif       1.553         9.3  1730
9      Tidak Puas  negatif       1.548         9.4  1730
11         Kecewa  negatif       1.524         8.7  1730
10       Frustasi  negatif       1.517         8.7  1730


In [60]:
# ── 4E. Overall Satisfaction per Kategori Layanan ─────────────
overall_summary = {}
OVERALL_LABEL_MAP = {
    'ovr_operasional':  'Jumlah & Waktu Operasional',
    'ovr_parkir':       'Ruang Parkir',
    'ovr_banking_hall': 'Banking Hall',
    'ovr_toilet':       'Toilet',
    'ovr_sekuriti':     'Sekuriti',
    'ovr_teller':       'Teller',
    'ovr_cs':           'Customer Service',
}
for col, label in OVERALL_LABEL_MAP.items():
    if col in df.columns:
        vals = df[col].dropna()
        if len(vals) > 0:
            overall_summary[label] = {
                'mean': round(vals.mean(), 3),
                'pct_satisfied': round((vals >= 5).mean() * 100, 1),
                'n': len(vals)
            }
overall_df = pd.DataFrame(overall_summary).T.reset_index()
overall_df.columns = ['kategori_layanan','mean_score','pct_puas','n']
overall_df.to_csv(f'{DATA_DIR}/overall_satisfaction.csv', index=False)
print('Overall Satisfaction per Kategori:')
print(overall_df.sort_values('mean_score', ascending=False))

Overall Satisfaction per Kategori:
             kategori_layanan  mean_score  pct_puas       n
4                    Sekuriti       5.908      99.8  1730.0
5                      Teller       5.896      99.5  1069.0
6            Customer Service       5.877      99.2  1046.0
3                      Toilet       5.873      99.2  1730.0
2                Banking Hall       5.850      99.0  1730.0
0  Jumlah & Waktu Operasional       5.839      99.0  1730.0
1                Ruang Parkir       5.793      98.4  1730.0


In [61]:
# ── 4F. Digitalisasi Summary ──────────────────────────────────
DIGI_MAP = {
    'T_SL1_1': ('Smart Tab','ada'),
    'T_SL1_2': ('Smart Tab','berfungsi'),
    'T_SL1_5': ('Digital Signage','ada'),
    'T_SL1_6': ('Digital Signage','berfungsi'),
    'T_SL1_7': ('Smart Table','ada'),
    'T_SL1_8': ('Smart Table','berfungsi'),
}
digi_records = []
for col, (label, tipe) in DIGI_MAP.items():
    if col in df.columns:
        vals = df[col].dropna()
        vals_str = vals.astype(str).str.strip()
        pct_ya = round((vals_str.str.lower().isin(['ya','1','ada'])).mean() * 100, 1)
        digi_records.append({'fasilitas': label, 'tipe': tipe, 'pct_ya': pct_ya, 'n': len(vals)})
digi_df = pd.DataFrame(digi_records)
digi_df.to_csv(f'{DATA_DIR}/digitalisasi.csv', index=False)
print('Digitalisasi Cabang:')
print(digi_df)

Digitalisasi Cabang:
         fasilitas       tipe  pct_ya     n
0        Smart Tab        ada     0.0  1730
1        Smart Tab  berfungsi     0.0  1730
2  Digital Signage        ada     0.0  1730
3  Digital Signage  berfungsi     0.0  1730
4      Smart Table        ada     0.0  1730
5      Smart Table  berfungsi     0.0  1730


In [62]:
# ── 4G. NPS Competitor ────────────────────────────────────────
def nps_score_col(series):
    s = pd.to_numeric(series, errors='coerce').dropna()
    if len(s) == 0: return np.nan
    return float(round(((s >= 9).sum() - (s <= 6).sum()) / len(s) * 100, 1))

nps_comp_records = [
    {'bank': 'Bank XYZ',     'nps_score': nps_score_col(df['G1A'])},
    {'bank': 'Kompetitor B', 'nps_score': nps_score_col(df['G1B'])},
    {'bank': 'Kompetitor C', 'nps_score': nps_score_col(df['G1C'])},
    {'bank': 'Kompetitor D', 'nps_score': nps_score_col(df['G1D'])},
]

nps_comp_df = pd.DataFrame(nps_comp_records).dropna()
nps_comp_df.to_csv('../data/nps_competitor.csv', index=False)
print(f'NPS Competitor: {len(nps_comp_df)} bank')
print(nps_comp_df)

NPS Competitor: 2 bank
           bank  nps_score
0      Bank XYZ       48.0
2  Kompetitor C        9.8


## 5. Export Data Master

In [63]:
# ── Export data master (baris per responden) ──────────────────
# Kolom yang dibutuhkan dashboard
EXPORT_COLS = [
    COL_ID, COL_PROV, COL_KOTA, COL_CABANG, COL_PANEL,
    COL_GENDER, COL_AGE_RAW, COL_AGE_GRP,
    COL_LAMA_NAS, COL_FREK,
    COL_NIKAH, COL_ANAK, COL_PENDIDIK, COL_PEKERJAAN,
    COL_PENGELUAR, COL_PENGHASIL,
    # KPI numerik
    'nps_num', 'csi_num', 'loyalty_num', 'nps_segment',
    # Emotion
    'emotion_positive_score', 'emotion_negative_score', 'emotion_net',
    # Segmen
    'customer_segment', 'lama_num', 'frek_num',
    # Overall layanan
] + list(OVERALL_COLS.keys())
# Tambahkan emotion individual
for c in ALL_EMOTION_COLS:
    nc = c + '_num'
    if nc in df.columns:
        EXPORT_COLS.append(nc)
# Filter hanya kolom yang ada
EXPORT_COLS = [c for c in EXPORT_COLS if c in df.columns]
df_export = df[EXPORT_COLS].copy()
# Rename untuk clarity
df_export.rename(columns={
    COL_PROV: 'provinsi', COL_KOTA: 'kota', COL_CABANG: 'cabang',
    COL_PANEL: 'panel', COL_GENDER: 'gender',
    COL_AGE_RAW: 'usia', COL_AGE_GRP: 'usia_group',
    COL_LAMA_NAS: 'lama_nasabah', COL_FREK: 'frekuensi_transaksi',
    COL_NIKAH: 'status_nikah', COL_ANAK: 'jumlah_anak',
    COL_PENDIDIK: 'pendidikan', COL_PEKERJAAN: 'pekerjaan',
    COL_PENGELUAR: 'pengeluaran', COL_PENGHASIL: 'penghasilan',
}, inplace=True)
df_export.to_csv(f'{DATA_DIR}/processed_bankxyz.csv', index=False)
print(f'Export selesai: {df_export.shape}')
print(f'File: data/processed_bankxyz.csv')
print(f'\nKolom yang diekspor: {len(df_export.columns)}')
print(df_export.dtypes.head(20))

Export selesai: (1730, 49)
File: data/processed_bankxyz.csv

Kolom yang diekspor: 49
SERIAL                   int64
provinsi                   str
kota                       str
cabang                     str
panel                      str
gender                     str
usia                     int64
usia_group                 str
lama_nasabah               str
frekuensi_transaksi        str
status_nikah               str
jumlah_anak                str
pendidikan                 str
pekerjaan                  str
pengeluaran                str
penghasilan                str
nps_num                float64
csi_num                float64
loyalty_num            float64
nps_segment                str
dtype: object


In [64]:
# ── Summary semua output ──────────────────────────────────────
import os
output_files = [
    (f'{DATA_DIR}/processed_bankxyz.csv', 'Data master per responden'),
    (f'{DATA_DIR}/agg_branch.csv', 'Agregasi per cabang'),
    (f'{DATA_DIR}/agg_provinsi.csv', 'Agregasi per provinsi'),
    (f'{DATA_DIR}/agg_gender.csv', 'Agregasi per gender'),
    (f'{DATA_DIR}/agg_usia.csv', 'Agregasi per usia'),
    (f'{DATA_DIR}/agg_panel.csv', 'Agregasi Teller vs CS'),
    (f'{DATA_DIR}/agg_segmen.csv', 'Segmentasi nasabah'),
    (f'{DATA_DIR}/ipa_matrix.csv', 'IPA Matrix semua touchpoint'),
    (f'{DATA_DIR}/emotion_summary.csv', 'Emotion summary XYZ'),
    (f'{DATA_DIR}/brand_perception.csv', 'Brand perception XYZ vs Kompetitor'),
    (f'{DATA_DIR}/overall_satisfaction.csv', 'Overall satisfaction per kategori'),
    (f'{DATA_DIR}/digitalisasi.csv', 'Digitalisasi cabang'),
]
print('=== OUTPUT FILES ===')
for path, desc in output_files:
    if os.path.exists(path):
        size = os.path.getsize(path)
        rows = sum(1 for _ in open(path)) - 1
        print(f'  ✓ {path:<45} {rows:>5} baris  ({desc})')
    else:
        print(f'  ✗ {path} — TIDAK ADA')

=== OUTPUT FILES ===
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/processed_bankxyz.csv  1730 baris  (Data master per responden)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_branch.csv   128 baris  (Agregasi per cabang)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_provinsi.csv    14 baris  (Agregasi per provinsi)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_gender.csv     2 baris  (Agregasi per gender)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_usia.csv     8 baris  (Agregasi per usia)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_panel.csv     2 baris  (Agregasi Teller vs CS)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/agg_segmen.csv     5 baris  (Segmentasi nasabah)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/ipa_matrix.csv   110 baris  (IPA Matrix semua touchpoint)
  ✓ C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data/emotion_summary.csv    16 baris  (Emotion sum